In [ ]:
# 数据处理库
import sys
import json
import logging
import pandas as pd
import numpy as np
from dataset_deduplicate_utils import*
from configs import DatasetDeduplicateConfig

In [ ]:
# 数据集基本信息参数传递
property_col = DatasetDeduplicateConfig._property
ref_col = DatasetDeduplicateConfig._reference
state_var_col = DatasetDeduplicateConfig._state_var

# 动态构建 essential_cols
essential_cols = ['c1', 'c2', 'x1', 'x2']
if property_col: essential_cols.append(property_col)
if ref_col: essential_cols.append(ref_col)
if state_var_col: essential_cols.append(state_var_col)

# 定义组分名称列 (使用 SMILES 作为唯一标识)
str_cols = [
    'c1_ismiles',
    'c2_ismiles',
    'c3_ismiles'
]

# 定义组分比例列
num_cols = [
    'x1',
    'x2', 
    'x3'
]

# 中间结果输出目录
interim_output_dir = DatasetDeduplicateConfig.DEDUPLICATE_DIR

# 定义处理完后要删除的辅助列 (例如原始的 c1, c2 名称列，如果不再需要)
drop_columns = ['c1', 'c2', 'c3']

In [ ]:
# 1. 读取文件
print("\n正在加载原始数据集...")
original_df = pd.read_excel(DatasetDeduplicateConfig.DATASET_WITH_IDS)
# =========================================================
# 2. 【阶段一】调用 utils 中的体检与重复项打标函数
# =========================================================
print("\n>>> 启动阶段一：识别重复数据 <<<")
intermediate_df = identify_and_export_duplicates(
    original_df=original_df,
    essential_cols=essential_cols,
    str_cols=str_cols,       # 传入组分标识列
    num_cols=num_cols,       # 传入比例列
    output_dir=interim_output_dir,
    ref_col=ref_col,         # 动态参考文献列名
    state_var_cols=state_var_col   
)

# 安全拦截：如果数据在阶段一被全部洗掉了，直接终止
if intermediate_df is None or intermediate_df.empty:
    print("❌ 数据清洗后无有效数据，程序终止。请检查输入数据质量。")
    import sys
    sys.exit(1)

In [ ]:
result_df = fuse_and_assemble_dataset(
    data_df=intermediate_df,
    property_col=property_col,
    drop_columns=drop_columns,
    ref_col=ref_col,
    agg_method='mean'
)

result_df.to_excel(DatasetDeduplicateConfig.DATASET_DEDUPLICATED, index=False)